# Practical No. 05: Image Generation using Pretrained Stable Diffusion

**Aim:** Implement a pretrained Stable Diffusion model using the Hugging Face `diffusers` library to generate images from text prompts, and analyze the impact of **inference steps**, **guidance scale** and **sampling methods (schedulers)** on image quality.

---
### Theory in brief
* **Stable Diffusion** is a *latent diffusion model*. A VAE compresses a 512×512 image into a 64×64×4 latent; a U-Net learns to remove noise from that latent, conditioned on a text embedding produced by a CLIP text encoder. Finally the VAE decoder converts the denoised latent back to pixels.
* **Inference steps** – number of denoising iterations. More steps = finer integration of the reverse diffusion process (better detail, slower).
* **Guidance scale (Classifier-Free Guidance, CFG)** – `noise = uncond + s · (cond − uncond)`. Higher `s` forces the image to follow the prompt more strongly but reduces diversity and can cause over-saturation / artifacts.
* **Scheduler / sampler** – the numerical solver used for the reverse process (PNDM, DDIM, Euler, Euler-Ancestral, DPM-Solver++, LMS, UniPC …). Different solvers converge at different speeds.

### Notebook outline
1. Setup & model loading
2. Basic text-to-image generation
3. Experiment 1 – effect of inference steps
4. Experiment 2 – effect of guidance scale
5. Experiment 3 – effect of sampling methods (schedulers)
6. Steps × Guidance grid
7. Quantitative evaluation (CLIP score + time)
8. Observations & conclusion

> **Runtime:** use a GPU (Colab → *Runtime → Change runtime type → T4 GPU*). The notebook also runs on CPU in `QUICK_MODE`, but it is very slow.

## 1. Setup

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate safetensors scipy matplotlib pandas

In [ ]:
import time, warnings, gc
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from diffusers import (
    StableDiffusionPipeline,
    PNDMScheduler, DDIMScheduler, EulerDiscreteScheduler,
    EulerAncestralDiscreteScheduler, DPMSolverMultistepScheduler,
    LMSDiscreteScheduler, UniPCMultistepScheduler, HeunDiscreteScheduler,
)

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
print(f"Device: {DEVICE} | dtype: {DTYPE}")
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ---------------- Global configuration ----------------
MODEL_ID   = "stable-diffusion-v1-5/stable-diffusion-v1-5"   # pretrained SD v1.5
SEED       = 42                                              # fixed seed => fair comparisons
QUICK_MODE = (DEVICE == "cpu")                               # shrink everything on CPU

IMG_SIZE   = 256 if QUICK_MODE else 512                      # SD 1.5 is trained at 512x512

PROMPT = ("a majestic lion standing on a rocky hill at sunset, "
          "golden hour lighting, highly detailed, 4k, photorealistic")
NEGATIVE_PROMPT = "blurry, low quality, distorted, deformed, ugly, watermark, text"

# Experiment settings
STEPS_LIST    = [5, 10, 20, 30, 50]    if not QUICK_MODE else [3, 5, 10]
GUIDANCE_LIST = [1.0, 3.0, 7.5, 12.0, 20.0] if not QUICK_MODE else [1.0, 7.5, 15.0]
SCHED_STEPS   = 25 if not QUICK_MODE else 8
print("Image size:", IMG_SIZE, "| steps:", STEPS_LIST, "| guidance:", GUIDANCE_LIST)

### 1.1 Load the pretrained Stable Diffusion pipeline

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    safety_checker=None,           # avoid black images on false positives (research / lab use only)
    requires_safety_checker=False,
)
pipe = pipe.to(DEVICE)
pipe.enable_attention_slicing()    # lowers VRAM use
pipe.set_progress_bar_config(disable=True)

BASE_SCHEDULER_CONFIG = pipe.scheduler.config   # keep so we can swap samplers later
print("Default scheduler :", pipe.scheduler.__class__.__name__)
print("UNet parameters   : %.1f M" % (sum(p.numel() for p in pipe.unet.parameters()) / 1e6))
print("VAE parameters    : %.1f M" % (sum(p.numel() for p in pipe.vae.parameters()) / 1e6))
print("Text encoder      : %.1f M" % (sum(p.numel() for p in pipe.text_encoder.parameters()) / 1e6))

### 1.2 Helper functions

In [ ]:
def set_scheduler(name):
    """Swap the sampler while re-using the pretrained weights."""
    registry = {
        "PNDM":            lambda c: PNDMScheduler.from_config(c),
        "DDIM":            lambda c: DDIMScheduler.from_config(c),
        "Euler":           lambda c: EulerDiscreteScheduler.from_config(c),
        "Euler Ancestral": lambda c: EulerAncestralDiscreteScheduler.from_config(c),
        "DPM++ 2M":        lambda c: DPMSolverMultistepScheduler.from_config(c),
        "DPM++ 2M Karras": lambda c: DPMSolverMultistepScheduler.from_config(c, use_karras_sigmas=True),
        "LMS":             lambda c: LMSDiscreteScheduler.from_config(c),
        "Heun":            lambda c: HeunDiscreteScheduler.from_config(c),
        "UniPC":           lambda c: UniPCMultistepScheduler.from_config(c),
    }
    pipe.scheduler = registry[name](BASE_SCHEDULER_CONFIG)
    return pipe.scheduler


def generate(prompt=PROMPT, steps=30, guidance=7.5, seed=SEED,
             negative_prompt=NEGATIVE_PROMPT, size=None):
    """Generate ONE image with a fixed seed. Returns (PIL image, seconds)."""
    size = size or IMG_SIZE
    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    out = pipe(prompt=prompt, negative_prompt=negative_prompt,
               num_inference_steps=steps, guidance_scale=guidance,
               height=size, width=size, generator=gen)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    return out.images[0], time.time() - t0


def show_grid(images, titles, ncols=None, suptitle=None, figsize_per=3.4):
    n = len(images)
    ncols = ncols or n
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_per * ncols, figsize_per * nrows + 0.6))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for ax, im, t in zip(axes, images, titles):
        ax.imshow(im)
        ax.set_title(t, fontsize=10)
    if suptitle:
        fig.suptitle(suptitle, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 2. Basic Text-to-Image Generation
Using the default settings: 30 steps, guidance scale 7.5, DPM++ 2M sampler.

In [ ]:
set_scheduler("DPM++ 2M")
img, t = generate(PROMPT, steps=30 if not QUICK_MODE else 10, guidance=7.5)
plt.figure(figsize=(6, 6)); plt.imshow(img); plt.axis("off")
plt.title(f"Prompt: {PROMPT[:60]}...\nTime: {t:.1f}s", fontsize=10); plt.show()
img.save("basic_generation.png")

### 2.1 Generate several different prompts
Shows the versatility of the model (same settings, different text).

In [ ]:
prompts = [
    "an astronaut riding a horse on mars, cinematic lighting, highly detailed",
    "a cozy wooden cabin in a snowy forest at night, warm lights, digital painting",
    "a futuristic city skyline with flying cars, neon lights, cyberpunk, 4k",
    "a bowl of fresh fruits on a wooden table, studio lighting, macro photography",
]
set_scheduler("DPM++ 2M")
imgs, ttl = [], []
for p in prompts:
    im, t = generate(p, steps=25 if not QUICK_MODE else 8, guidance=7.5)
    imgs.append(im); ttl.append(p[:38] + "...\n(%.1fs)" % t)
show_grid(imgs, ttl, ncols=2, suptitle="Different text prompts, same settings")

## 3. Experiment 1 – Effect of Number of Inference Steps
Everything is held constant (prompt, seed, guidance = 7.5, sampler = **DDIM**) and only `num_inference_steps` changes.
Because the seed is the same, all images begin from the *same initial noise*, so differences are caused purely by the number of denoising steps.

In [ ]:
set_scheduler("DDIM")
step_imgs, step_titles, step_rows = [], [], []
for s in STEPS_LIST:
    im, t = generate(PROMPT, steps=s, guidance=7.5)
    step_imgs.append(im)
    step_titles.append(f"steps = {s}\n{t:.1f}s")
    step_rows.append({"steps": s, "time_s": t})
    print(f"steps={s:>3} -> {t:.2f}s")

show_grid(step_imgs, step_titles, suptitle="Effect of inference steps (DDIM, guidance = 7.5, seed fixed)")

In [ ]:
df_steps = pd.DataFrame(step_rows)
plt.figure(figsize=(5.5, 3.5))
plt.plot(df_steps.steps, df_steps.time_s, "o-")
plt.xlabel("Inference steps"); plt.ylabel("Generation time (s)")
plt.title("Time grows ~linearly with steps"); plt.grid(alpha=.3); plt.show()
df_steps

**Expected observation:** with very few steps (5–10) the image is noisy / blurry with poorly formed structure. Quality improves quickly up to ~20–30 steps, after which gains become marginal while time keeps increasing linearly (**diminishing returns**).

## 4. Experiment 2 – Effect of Guidance Scale (CFG)
Prompt, seed, steps (30) and sampler (DDIM) are fixed; only `guidance_scale` varies.

In [ ]:
set_scheduler("DDIM")
FIXED_STEPS = 30 if not QUICK_MODE else 10
g_imgs, g_titles = [], []
for g in GUIDANCE_LIST:
    im, t = generate(PROMPT, steps=FIXED_STEPS, guidance=g)
    g_imgs.append(im); g_titles.append(f"guidance = {g}")

show_grid(g_imgs, g_titles, suptitle=f"Effect of guidance scale (DDIM, steps = {FIXED_STEPS}, seed fixed)")

**Expected observation**

| Guidance | Behaviour |
|---|---|
| ~1 | Almost ignores the prompt – vague, low contrast, creative but unrelated |
| 3–5 | Loosely follows prompt, natural-looking, more diverse |
| 7–9 | Sweet spot – good prompt adherence and image quality |
| 12–20+ | Over-saturated colors, harsh contrast, artifacts, loss of realism |

## 5. Experiment 3 – Effect of Sampling Methods (Schedulers)
Same prompt, seed, guidance = 7.5 and the **same number of steps**; only the scheduler changes.

In [ ]:
SCHEDULERS = ["PNDM", "DDIM", "Euler", "Euler Ancestral", "DPM++ 2M", "DPM++ 2M Karras", "LMS", "UniPC"]
if QUICK_MODE:
    SCHEDULERS = ["PNDM", "DDIM", "Euler", "DPM++ 2M"]

s_imgs, s_titles, s_rows = [], [], []
for name in SCHEDULERS:
    set_scheduler(name)
    im, t = generate(PROMPT, steps=SCHED_STEPS, guidance=7.5)
    s_imgs.append(im); s_titles.append(f"{name}\n{t:.1f}s")
    s_rows.append({"scheduler": name, "steps": SCHED_STEPS, "time_s": t})
    print(f"{name:<18} {t:.2f}s")

show_grid(s_imgs, s_titles, ncols=4, suptitle=f"Different samplers ({SCHED_STEPS} steps, guidance = 7.5)")
pd.DataFrame(s_rows)

### 5.1 Convergence of samplers at low step counts
Good samplers (DPM++, UniPC, Euler) produce usable images in ~10–15 steps, whereas older ones (PNDM, LMS, DDIM) need more.

In [ ]:
LOW_STEPS = 8 if not QUICK_MODE else 4
cmp_names = ["PNDM", "DDIM", "Euler", "DPM++ 2M"] if QUICK_MODE else ["PNDM", "DDIM", "Euler Ancestral", "DPM++ 2M Karras", "UniPC"]
imgs, ttl = [], []
for name in cmp_names:
    set_scheduler(name)
    im, t = generate(PROMPT, steps=LOW_STEPS, guidance=7.5)
    imgs.append(im); ttl.append(f"{name}\n{LOW_STEPS} steps")
show_grid(imgs, ttl, ncols=len(imgs), suptitle=f"Low step-count behaviour ({LOW_STEPS} steps)", figsize_per=3.0)

## 6. Combined grid: Steps × Guidance scale
Shows the interaction between the two parameters (sampler: DPM++ 2M).

In [ ]:
set_scheduler("DPM++ 2M")
grid_steps = [5, 15, 30] if not QUICK_MODE else [3, 8]
grid_guid  = [2.0, 7.5, 15.0]
fig, axes = plt.subplots(len(grid_steps), len(grid_guid), figsize=(3.4 * len(grid_guid), 3.4 * len(grid_steps)))
for i, s in enumerate(grid_steps):
    for j, g in enumerate(grid_guid):
        im, _ = generate(PROMPT, steps=s, guidance=g)
        axes[i, j].imshow(im); axes[i, j].axis("off")
        axes[i, j].set_title(f"steps={s}, cfg={g}", fontsize=10)
fig.suptitle("Steps × Guidance scale", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

## 7. Quantitative Evaluation (CLIP Score + Time)
Visual inspection is subjective, so we also compute the **CLIP score** – cosine similarity (×100) between the CLIP embedding of the generated image and the prompt. A higher value means better *text–image alignment*. (It measures prompt adherence, not photorealism.)

In [ ]:
from transformers import CLIPModel, CLIPProcessor

clip_id = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_id).to(DEVICE).eval()
clip_proc  = CLIPProcessor.from_pretrained(clip_id)

@torch.no_grad()
def clip_score(image, text):
    inputs = clip_proc(text=[text], images=image, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    out = clip_model(**inputs)
    img_e = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True)
    txt_e = out.text_embeds  / out.text_embeds.norm(dim=-1, keepdim=True)
    return float((img_e * txt_e).sum() * 100)

EVAL_PROMPTS = [
    "a majestic lion standing on a rocky hill at sunset, golden hour lighting, highly detailed",
    "an astronaut riding a horse on mars, cinematic lighting",
    "a cozy wooden cabin in a snowy forest at night, warm lights",
]
if QUICK_MODE:
    EVAL_PROMPTS = EVAL_PROMPTS[:2]
SEEDS = [42, 7] if not QUICK_MODE else [42]

### 7.1 CLIP score vs. inference steps

In [ ]:
set_scheduler("DDIM")
rows = []
for s in STEPS_LIST:
    for p in EVAL_PROMPTS:
        for sd in SEEDS:
            im, t = generate(p, steps=s, guidance=7.5, seed=sd)
            rows.append({"steps": s, "clip": clip_score(im, p), "time_s": t})
df1 = pd.DataFrame(rows).groupby("steps").mean().reset_index()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(df1.steps, df1.clip, "o-", color="tab:blue"); ax[0].set_xlabel("Inference steps"); ax[0].set_ylabel("CLIP score"); ax[0].set_title("CLIP score vs steps"); ax[0].grid(alpha=.3)
ax[1].plot(df1.steps, df1.time_s, "o-", color="tab:red");  ax[1].set_xlabel("Inference steps"); ax[1].set_ylabel("Time (s)");  ax[1].set_title("Time vs steps");       ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()
df1.round(2)

### 7.2 CLIP score vs. guidance scale

In [ ]:
set_scheduler("DDIM")
rows = []
for g in GUIDANCE_LIST:
    for p in EVAL_PROMPTS:
        for sd in SEEDS:
            im, _ = generate(p, steps=FIXED_STEPS, guidance=g, seed=sd)
            rows.append({"guidance": g, "clip": clip_score(im, p)})
df2 = pd.DataFrame(rows).groupby("guidance").agg(clip=("clip", "mean"), std=("clip", "std")).reset_index()

plt.figure(figsize=(6, 3.6))
plt.errorbar(df2.guidance, df2.clip, yerr=df2["std"].fillna(0), fmt="o-", capsize=3)
plt.xlabel("Guidance scale"); plt.ylabel("CLIP score"); plt.title("CLIP score vs guidance scale"); plt.grid(alpha=.3); plt.show()
df2.round(2)

### 7.3 CLIP score & time per sampler

In [ ]:
rows = []
for name in SCHEDULERS:
    set_scheduler(name)
    for p in EVAL_PROMPTS:
        for sd in SEEDS:
            im, t = generate(p, steps=SCHED_STEPS, guidance=7.5, seed=sd)
            rows.append({"scheduler": name, "clip": clip_score(im, p), "time_s": t})
df3 = pd.DataFrame(rows).groupby("scheduler", sort=False).mean().reset_index()

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].bar(df3.scheduler, df3.clip, color="tab:blue");  ax[0].set_title("CLIP score by sampler"); ax[0].set_ylim(df3.clip.min() - 1, df3.clip.max() + 1)
ax[1].bar(df3.scheduler, df3.time_s, color="tab:red"); ax[1].set_title("Time by sampler (s)")
for a in ax:
    a.tick_params(axis="x", rotation=45); a.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.show()
df3.round(2)

## 8. Optional: Negative prompt effect
A quick extra demonstration of how the negative prompt steers the output away from unwanted features.

In [ ]:
set_scheduler("DPM++ 2M")
st = 25 if not QUICK_MODE else 8
im_no,  _ = generate(PROMPT, steps=st, guidance=7.5, negative_prompt="")
im_yes, _ = generate(PROMPT, steps=st, guidance=7.5, negative_prompt=NEGATIVE_PROMPT)
show_grid([im_no, im_yes], ["No negative prompt", "With negative prompt"], suptitle="Negative prompt effect")

## 9. Observations & Conclusion

**Inference steps**
* Too few steps (≤ 10) leave residual noise and blurry / malformed structure.
* Quality saturates around 20–30 steps with a good sampler; beyond ~50 steps visual gains are negligible but time grows linearly.

**Guidance scale**
* Low (1–3): creative but weak adherence to the prompt (low CLIP score).
* Medium (7–9): best balance of prompt fidelity, detail and realism.
* High (≥ 15): over-saturated, high-contrast images with artifacts and reduced diversity; CLIP score stops improving or drops.

**Sampling methods**
* PNDM / DDIM / LMS are older solvers and need more steps for sharp results.
* Euler, DPM++ 2M (Karras), UniPC reach good quality in ~15–25 steps (faster convergence).
* Ancestral samplers (Euler a) inject fresh noise at each step, so outputs keep changing with step count and do not converge to a single image.
* Heun is more accurate per step but evaluates the U-Net twice per step (≈ 2× slower).

**Conclusion:** Stable Diffusion produces high-quality text-conditioned images, but results depend on the interplay of *steps*, *guidance scale* and *sampler*. A good default is **DPM++ 2M (Karras), 20–30 steps, guidance 7–8**, which gives a strong quality / speed trade-off.

> *Note:* Exact images and numbers differ by hardware, library version and seed; your own plots/values from the runs above should be used when writing the final report.

In [ ]:
# Free GPU memory (optional)
del pipe, clip_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()